# Task 2 — Bias Correction On vs Off

> **Ask.** Disable bias correction and plot the first twenty steps both ways
> (with and without). Report the number of steps after which the difference
> stops mattering.

### What bias correction is

Adam initialises $m_0 = v_0 = 0$. For the first steps the EMAs are therefore
biased *toward zero*. The fix:

$$\hat m_t = \frac{m_t}{1-\beta_1^{\,t}}, \qquad \hat v_t = \frac{v_t}{1-\beta_2^{\,t}}.$$

With bias correction **off** we use $m_t, v_t$ directly. The net effect on the
update is a multiplicative factor

$$
c(t) \;=\; \frac{\text{corrected step}}{\text{uncorrected step}}
       \;=\; \frac{1/(1-\beta_1^{\,t})}{\sqrt{1/(1-\beta_2^{\,t})}}
       \;=\; \frac{\sqrt{1-\beta_2^{\,t}}}{1-\beta_1^{\,t}}.
$$

$c(t)\to 1$ as $t\to\infty$; the question is *how fast*, and it depends almost
entirely on $\beta_2$.

In [ ]:
import sys, os, math, time, json, warnings
sys.path.insert(0, os.path.abspath("../src"))
warnings.filterwarnings("ignore")

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

from s11.utils import set_seed, get_device, savefig, plot_style, AIM_REPO
plot_style()
set_seed(1337)
DEVICE = get_device()
torch.set_float32_matmul_precision("high")
print(f"torch {torch.__version__} | device = {DEVICE} "
      f"| {torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'cpu'}")
print(f"Aim repo: {AIM_REPO}  (browse it later with:  uv run aim up)")

## 1. The correction factor $c(t)$ in closed form

We tabulate $c(t)$ for three $\beta_2$ values: `0.999` (Adam default), `0.99`
(this repo's training default) and `0.95` (nanoGPT / LLM default). "Stops
mattering" = $|c(t)-1| < \tau$; we use $\tau = 1\%$ and also report $5\%$.

In [ ]:
def c_factor(t, b1, b2):
    return math.sqrt(1 - b2**t) / (1 - b1**t)

B1 = 0.9

def steps_until_within(b2, tau, b1=B1, tmax=200000, hold=1000):
    """First t such that |c(t')-1| < tau for that t' AND stays there.

    c(t) is non-monotonic for small 1-b2 (it dips below 1 before climbing back),
    so we require the band to hold for `hold` consecutive steps, not just once."""
    run = 0
    for t in range(1, tmax):
        if abs(c_factor(t, b1, b2) - 1) < tau:
            run += 1
            if run == 1:
                first = t
            if run >= hold or t == tmax - 1:
                return first
        else:
            run = 0
    return None

rows = []
for b2 in [0.999, 0.99, 0.95]:
    for tau in [0.05, 0.01]:
        rows.append(dict(beta2=b2, tolerance=f"{tau:.0%}",
                         steps_until_within_tol=steps_until_within(b2, tau)))
tab = pd.DataFrame(rows)
print(tab.to_string(index=False))
print("\n(c(t) is NOT monotonic for beta2<=0.99 — it dips below 1 around t~10-15 "
      "then climbs back, so 'first touch' of the band is not 'stays in' the band.)")

print("\nc(t) for the first 20 steps:")
head = pd.DataFrame({"t": range(1, 21),
                     "c(t)  b2=0.999": [c_factor(t,B1,0.999) for t in range(1,21)],
                     "c(t)  b2=0.99":  [c_factor(t,B1,0.99)  for t in range(1,21)],
                     "c(t)  b2=0.95":  [c_factor(t,B1,0.95)  for t in range(1,21)]})
print(head.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 2. First twenty steps on a real weight

We take a genuine parameter from the **10.77M nanoGPT** — a slice of the first
block's MLP input matrix — and record the true gradient it receives on each of
the first 20 optimizer steps. Then we replay those exact 20 gradients through our
hand Adam **with** and **without** bias correction (everything else identical).

In [ ]:
from s11.model import GPT, GPTConfig
from s11.data import load_char_dataset
from s11.optim import run_adam_by_hand

data = load_char_dataset()
set_seed(1337)
cfg = GPTConfig(vocab_size=data.vocab_size, block_size=256)
model = GPT(cfg).to(DEVICE)
print(f"model params: {model.num_params()/1e6:.2f}M")

# choose one weight: element [0,0] of block 0's MLP fc weight
target_name = "transformer.h.0.mlp.c_fc.weight"
param = dict(model.named_parameters())[target_name]
idx = (0, 0)
w0 = param.detach()[idx].item()

opt = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.99), weight_decay=0.0)
gen = torch.Generator().manual_seed(0)
observed_grads = []
model.train()
for step in range(20):
    X, Y = data.get_batch("train", 32, 256, DEVICE, gen)
    _, loss = model(X, Y)
    opt.zero_grad(set_to_none=True); loss.backward()
    observed_grads.append(param.grad.detach()[idx].item())
    opt.step()

print(f"weight  {target_name}{list(idx)}  w0 = {w0:.6f}")
print("observed gradients (first 20 steps):")
print(np.array2string(np.array(observed_grads), precision=5, floatmode="fixed"))

In [ ]:
B2 = 0.99   # this repo's training default
with_bc    = run_adam_by_hand(w0, observed_grads, lr=1e-3, beta1=0.9, beta2=B2, bias_correction=True)
without_bc = run_adam_by_hand(w0, observed_grads, lr=1e-3, beta1=0.9, beta2=B2, bias_correction=False)

cmp = pd.DataFrame({
    "t":        [r["t"] for r in with_bc],
    "step_BC":  [r["step"] for r in with_bc],
    "step_noBC":[r["step"] for r in without_bc],
    "w_BC":     [r["param_after"] for r in with_bc],
    "w_noBC":   [r["param_after"] for r in without_bc],
})
cmp["|step ratio|"] = (cmp["step_BC"] / cmp["step_noBC"]).abs()
cmp["c(t) predicted"] = [c_factor(t, 0.9, B2) for t in cmp["t"]]
cmp["w gap"] = (cmp["w_BC"] - cmp["w_noBC"]).abs()
print(cmp.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

## 3. "When does the difference stop mattering?"

Two complementary readings:

* **Per-step** — when the update-size ratio $c(t)$ is within tolerance of 1.
* **Cumulative** — when the *weight trajectories* stop drifting apart (the gap
  plateaus, because later steps are near-identical).

In [ ]:
gap = np.array([abs(a["param_after"] - b["param_after"]) for a, b in zip(with_bc, without_bc)])
# gap relative to the weight's own movement so far (does the disagreement still matter?)
move_bc = np.abs(np.array([r["param_after"] for r in with_bc]) - w0) + 1e-12
rel_gap = gap / move_bc

print(f"this run used beta2 = {B2}\n")
print(f"{'beta2':>7} | {'within 5% (stays)':>18} | {'within 1% (stays)':>18}")
for b2 in [0.999, 0.99, 0.95]:
    print(f"{b2:>7} | {str(steps_until_within(b2,0.05)):>18} | {str(steps_until_within(b2,0.01)):>18}")
print(f"\nper-step   : c(t) enters & stays within 1% of 1 from step t = {steps_until_within(B2, 0.01)}")
print(f"cumulative : after 20 steps, |w_BC - w_noBC| = {gap[-1]:.2e}, "
      f"which is {rel_gap[-1]:.0%} of how far this weight has moved — still a real gap,")
print(f"             because with beta2={B2} the correction factor is still {c_factor(20,0.9,B2):.2f} at step 20.")

## 4. Charts — first 20 steps both ways

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
t20 = list(range(1, 21))

ax[0,0].plot(t20, [r["step"] for r in with_bc], "o-", label="bias correction ON")
ax[0,0].plot(t20, [r["step"] for r in without_bc], "s--", label="bias correction OFF")
ax[0,0].set_title("Per-step update $\\Delta_t$"); ax[0,0].set_xlabel("step"); ax[0,0].legend()

ax[0,1].plot(t20, [r["param_after"] for r in with_bc], "o-", label="ON")
ax[0,1].plot(t20, [r["param_after"] for r in without_bc], "s--", label="OFF")
ax[0,1].set_title("Weight trajectory (20 steps)"); ax[0,1].set_xlabel("step"); ax[0,1].legend()

ax[0,2].plot(t20, gap, "d-", color="crimson")
ax[0,2].set_title("|w_BC - w_noBC|  (cumulative gap)"); ax[0,2].set_xlabel("step")

for b2, mk in [(0.999,"o-"), (0.99,"s-"), (0.95,"^-")]:
    ax[1,0].plot(range(1,21), [c_factor(t,0.9,b2) for t in range(1,21)], mk, label=f"β2={b2}")
ax[1,0].axhline(1, ls=":", color="grey"); ax[1,0].axhspan(0.99,1.01, color="green", alpha=0.1)
ax[1,0].set_title("Correction factor c(t), first 20 steps"); ax[1,0].set_xlabel("step"); ax[1,0].legend()

tt = np.arange(1, 6001)
for b2, mk in [(0.999,"-"), (0.99,"-"), (0.95,"-")]:
    ax[1,1].plot(tt, [c_factor(t,0.9,b2) for t in tt], mk, label=f"β2={b2}")
ax[1,1].axhspan(0.99,1.01, color="green", alpha=0.1)
ax[1,1].set_xscale("log"); ax[1,1].set_title("c(t) to convergence (log x)")
ax[1,1].set_xlabel("step"); ax[1,1].legend()

ax[1,2].plot(t20, [1-0.99**t for t in t20], "s-", label=r"$1-\beta_2^t$ (β2=0.99)")
ax[1,2].plot(t20, [1-0.9**t for t in t20], "o-", label=r"$1-\beta_1^t$ (β1=0.9)")
ax[1,2].set_title("Why the gap: EMA warm-up"); ax[1,2].set_xlabel("step"); ax[1,2].legend()

plt.tight_layout()
savefig(fig, "t2_bias_correction.png")
plt.show()

## 5. Findings

**Reported number — "steps after which the difference stops mattering"**
(first step at which $|c(t)-1|$ enters the band *and stays*, since $c(t)$ is
non-monotonic for $\beta_2\le 0.99$):

| $\beta_2$ | within 5% | within 1% | context |
|---|---|---|---|
| 0.95  | **42** steps  | **76** steps | nanoGPT / LLM default |
| 0.99  | **232** steps | **390** steps | this repo's training default |
| 0.999 | **2 327** steps | **3 916** steps | Adam-paper default |

The horizon scales as $t^\star \approx \ln(0.0X)/\ln\beta_2$ — set by $\beta_2$
alone ($\beta_1$'s term has decayed by then). $\beta_1$'s own bias is gone by
~step 45 (within 1%).

**Interpretation**

- With correction **OFF**, step 1 uses $m_1 = (1-\beta_1)g_1 \approx 0.1\,g_1$ and
  $v_1=(1-\beta_2)g_1^2$, so the update is $\approx (1-\beta_1)/\sqrt{1-\beta_2}$
  of a unit step — for $\beta_2=0.99$ that ratio is exactly **1.0** (coincidence:
  $\sqrt{0.01}=0.1$), for $\beta_2=0.999$ it is **0.32** (much smaller), for
  $\beta_2=0.95$ it is **2.24** (much *larger*). So "bias correction off" is not
  uniformly a warmup — its sign depends on $\beta_2$.
- After the first ~10–15 steps $c(t)$ **dips below 1** (the $m$-EMA has caught up
  but the $v$-EMA has not, so $\sqrt{\hat v}$ is still too small → uncorrected step
  too *large*), then climbs back to 1 over hundreds/thousands of steps.
- Because it self-heals within tens-to-hundreds of steps, and real training uses
  an explicit warmup anyway, **bias correction is nearly irrelevant to the final
  loss** of a multi-thousand-step run — but it is very cheap and removes a
  confound, so keep it on.
- The horizon scales with $\beta_2$: the slower the second-moment EMA, the longer
  the correction is doing real work.